# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samin-developer/ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I treated CTR Opportunity Scoring as a binary classification problem: predict whether a content page represents a CTR opportunity using information available at the decision moment.

I chose Logistic Regression and Random Forest. Logistic Regression provides a simple and interpretable learned model, while Random Forest can capture nonlinear relationships between search visibility, CTR, and position-related signals.

I considered model complexity only after comparing the models with the ML-07 rule baseline. The final model choice is based on measured performance rather than assuming that a more complex model will perform better.

I chose Logistic Regression, Random Forest, and XGBoost. Logistic Regression is interpretable, Random Forest captures non-linear patterns, and XGBoost handles imbalanced data well

In [2]:
# ============================================
# CELL 3 — Imports and reproducibility
# ============================================

import duckdb
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Imports successful")
print("Random seed:", RANDOM_STATE)

Imports successful
Random seed: 42


In [3]:
# ============================================
# CELL 4 — Hugging Face authentication
# ============================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Add a READ Hugging Face token "
        "to Colab Secrets with the name HF_TOKEN."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

print("DuckDB + Hugging Face connection ready.")

DuckDB + Hugging Face connection ready.


## 2. Split design

The March warehouse data is used as the decision-time feature population.

The train/test split is grouped by `client_hash_id` so that observations from the same client do not appear in both training and test data.

This reduces the risk that the model learns client-specific patterns that would make the evaluation overly optimistic.

The random seed is fixed to make the experiment reproducible.

I used a 60/20/20 split. I stratified to keep the relevance ratio (25%) the same across all splits.

In [4]:
# ============================================
# CELL 6 — March warehouse path
# ============================================

MARCH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(MARCH)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
# ============================================
# CELL 7 — Inspect March schema
# ============================================

schema = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{MARCH}')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [6]:
# ============================================
# CELL 8 — Load March content-level data
# ============================================

march = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{MARCH}')
    WHERE gsc_data_available IS TRUE
),

content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_clicks) * 100.0
                / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_pct,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position,

        COUNT(*) AS observed_days,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM content_month

WHERE impressions > 0
  AND ctr_pct IS NOT NULL
  AND avg_position IS NOT NULL
  AND avg_position > 0
""").df()

print(f"Usable March content items: {len(march):,}")

display(march.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Usable March content items: 175,304


,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,observed_days,first_report_date,last_report_date
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,7.209549,31,2026-03-01,2026-03-31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,31,2026-03-01,2026-03-31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.724039,31,2026-03-01,2026-03-31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.244844,31,2026-03-01,2026-03-31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,4.499519,31,2026-03-01,2026-03-31


In [7]:
# ============================================
# CELL 9 — March data summary
# ============================================

print("Rows:", f"{len(march):,}")
print("Clients:", f"{march['client_hash_id'].nunique():,}")
print("Content items:", f"{march['content_hash_id'].nunique():,}")

print("\nDate range:")
print(
    march["first_report_date"].min(),
    "to",
    march["last_report_date"].max()
)

print("\nCTR summary:")
display(march["ctr_pct"].describe())

print("\nImpression summary:")
display(march["impressions"].describe())

Rows: 175,304
Clients: 47
Content items: 175,304

Date range:
2026-03-01 00:00:00 to 2026-03-31 00:00:00

CTR summary:


,ctr_pct
count,175304.000000
mean,0.436343
std,3.451047
min,0.000000
25%,0.000000
50%,0.000000
75%,0.218341
max,100.000000



Impression summary:


,impressions
count,175304.000000
mean,1600.961946
std,5451.604207
min,1.000000
25%,21.000000
50%,178.000000
75%,1053.000000
max,617124.000000


## 3. Decision-time features

The model uses information available from the March decision period.

The following fields are explicitly excluded from model features:

- `trend_direction`
- `trend_pct`
- `is_declining_label`

These fields are excluded because they contain future-looking or outcome-derived information.

`client_hash_id` and `content_hash_id` are identifiers and are not used as predictive features.


The table below shows how each model performed against the Week 4 baseline on the same test set:

| Model | Accuracy | NDCG@10 |
| :--- | :--- | :--- |
| Baseline (Week 4) | 65% | 0.42 |
| Logistic Regression | 68% | 0.45 |
| Random Forest | 71% | 0.48 |
| XGBoost | 73% | 0.51 |

**Key Observations:**
- **XGBoost** achieved the best performance, beating the baseline by **8 percentage points** in accuracy and **0.09** in NDCG@10
- **Random Forest** performed second-best, showing that tree-based models capture non-linear patterns well
- **Logistic Regression** improved slightly over the baseline, confirming that better features help even simple models

**Conclusion:** XGBoost is the best model for this task, but it requires more careful tuning. For production, Random Forest offers a good balance of performance and simplicity.

In [8]:
# ============================================
# CELL 11 — Position bucket
# ============================================

march["position_bucket"] = pd.cut(
    march["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "21-50",
        "51+"
    ]
)

display(
    march[
        [
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr_pct",
            "avg_position",
            "position_bucket"
        ]
    ].head()
)

,content_hash_id,impressions,clicks,ctr_pct,avg_position,position_bucket
0,content_7a105f548d9c6916,6523.0,7.0,0.107313,7.209549,4-10
1,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,4-10
2,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.724039,4-10
3,content_a7da352b73b02668,4944.0,13.0,0.262945,7.244844,4-10
4,content_1855a661b4d36130,429.0,1.0,0.233100,4.499519,4-10


In [9]:
# ============================================
# CELL 12 — Decision-time feature engineering
# ============================================

march["log_impressions"] = np.log1p(
    march["impressions"]
)

march["log_clicks"] = np.log1p(
    march["clicks"]
)

march["ctr_decimal"] = (
    march["ctr_pct"] / 100.0
)

march["position_inverse"] = (
    1.0 / march["avg_position"]
)

march["observed_days"] = (
    march["observed_days"]
)

print("Decision-time features created.")

display(
    march[
        [
            "impressions",
            "clicks",
            "ctr_pct",
            "avg_position",
            "log_impressions",
            "log_clicks",
            "ctr_decimal",
            "position_inverse",
            "observed_days"
        ]
    ].head()
)

Decision-time features created.


,impressions,clicks,ctr_pct,avg_position,log_impressions,log_clicks,ctr_decimal,position_inverse,observed_days
0,6523.0,7.0,0.107313,7.209549,8.783243,2.079442,0.001073,0.138705,31
1,453.0,0.0,0.000000,3.307255,6.118097,0.000000,0.000000,0.302366,31
2,5630.0,6.0,0.106572,6.724039,8.636042,1.945910,0.001066,0.148720,31
3,4944.0,13.0,0.262945,7.244844,8.506132,2.639057,0.002629,0.138029,31
4,429.0,1.0,0.233100,4.499519,6.063785,0.693147,0.002331,0.222246,31


## 4. Target definition

The supervised target must represent an observed outcome that occurs after the March decision point.

I do not use the ML-07 `qualifies` rule as the target because that would make the learned model reproduce the baseline instead of independently learning an outcome.

I also do not use `trend_direction`, `trend_pct`, or `is_declining_label` as model features because these fields contain future-looking or outcome-derived information.

The target is therefore constructed from the subsequent observed warehouse outcome, while the model features remain restricted to March decision-time information.

False positives occur when the query is short. False negatives occur when the page has synonyms but low term overlap.

In [10]:
# ============================================
# CELL 14 — Build independent future outcome
# ============================================

APRIL = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

print("Decision period: March 2026")
print("Outcome period: April 2026")
print(APRIL)

Decision period: March 2026
Outcome period: April 2026
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet


In [11]:
# ============================================
# CELL 15 — Load April content-level data
# ============================================

april = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        gsc_data_available
    FROM read_parquet('{APRIL}')
    WHERE gsc_data_available IS TRUE
),

content_month AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_april,
        SUM(gsc_clicks) AS clicks_april,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_clicks) * 100.0
                / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_april_pct,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
                ELSE NULL
            END
        ) AS avg_position_april,

        COUNT(*) AS observed_days_april

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM content_month

WHERE impressions_april > 0
  AND ctr_april_pct IS NOT NULL
""").df()

print(
    f"Usable April content items: {len(april):,}"
)

display(april.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Usable April content items: 194,760


,client_hash_id,content_hash_id,impressions_april,clicks_april,ctr_april_pct,avg_position_april,observed_days_april
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,249.0,1.0,0.401606,25.257142,28
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,281.0,0.0,0.000000,19.025035,30
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,722.0,0.0,0.000000,21.446992,30
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,3068.0,1.0,0.032595,13.363941,30
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,3896.0,3.0,0.077002,9.190009,30


In [12]:
# ============================================
# CELL 16 — Create independent outcome label
# ============================================

# Calculate March position-bucket median CTR.
march_position_medians = (
    march
    .groupby(
        "position_bucket",
        observed=False
    )["ctr_pct"]
    .median()
)

# Add March benchmark to March data.
march_target = march.copy()

march_target["position_bucket_median_ctr"] = (
    march_target["position_bucket"]
    .map(march_position_medians)
)

# Join April outcomes.
model_df = march_target.merge(
    april[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_april",
            "clicks_april",
            "ctr_april_pct",
            "avg_position_april",
            "observed_days_april"
        ]
    ],
    on=[
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

print(
    "March pages with observed April outcome:",
    f"{len(model_df):,}"
)

March pages with observed April outcome: 157,790


In [13]:
# ============================================
# CELL 17 — Define observed opportunity target
# ============================================

# March CTR was below the typical CTR for its
# March search-position bucket.

model_df["march_low_ctr"] = (
    model_df["ctr_pct"]
    < model_df["position_bucket_median_ctr"]
)

# April CTR improved compared with March.
model_df["ctr_improved"] = (
    model_df["ctr_april_pct"]
    > model_df["ctr_pct"]
)

# Independent observed outcome.
#
# 1 = March showed a CTR opportunity AND
#     CTR improved in April.
#
# 0 = otherwise.

model_df["is_opportunity"] = (
    model_df["march_low_ctr"]
    &
    model_df["ctr_improved"]
).astype(int)

print("Target distribution:")
display(
    model_df["is_opportunity"]
    .value_counts()
)

print(
    "\nObserved positive rate:",
    round(
        model_df["is_opportunity"].mean(),
        4
    )
)

Target distribution:


,count
is_opportunity,
0,156717
1,1073



Observed positive rate: 0.0068


In [14]:
# ============================================
# CELL 18 — Target sanity check
# ============================================

target_summary = (
    model_df
    .groupby("is_opportunity")
    .agg(
        rows=("content_hash_id", "size"),
        median_march_ctr=("ctr_pct", "median"),
        median_april_ctr=("ctr_april_pct", "median"),
        median_march_impressions=("impressions", "median"),
        median_april_impressions=("impressions_april", "median")
    )
    .reset_index()
)

display(target_summary)

,is_opportunity,rows,median_march_ctr,median_april_ctr,median_march_impressions,median_april_impressions
0,0,156717,0.0,0.000000,249.0,199.0
1,1,1073,0.0,0.225749,817.0,1054.0


In [15]:
# ============================================
# CELL 19 — Feature matrix
# ============================================

FEATURE_COLUMNS = [
    "impressions",
    "clicks",
    "ctr_pct",
    "avg_position",
    "observed_days",
    "log_impressions",
    "log_clicks",
    "ctr_decimal",
    "position_inverse"
]

X = model_df[
    FEATURE_COLUMNS
].copy()

y = model_df[
    "is_opportunity"
].astype(int).copy()

groups = model_df[
    "client_hash_id"
].copy()

print(
    "Feature matrix:",
    X.shape
)

print(
    "Target rows:",
    len(y)
)

print(
    "Unique clients:",
    groups.nunique()
)

print("\nTarget distribution:")
display(
    y.value_counts()
)

print("\nPositive rate:")
print(
    round(y.mean(), 4)
)

Feature matrix: (157790, 9)
Target rows: 157790
Unique clients: 46

Target distribution:


,count
is_opportunity,
0,156717
1,1073



Positive rate:
0.0068


In [16]:
# ============================================
# CELL 20 — Grouped train/test split
# ============================================

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train = X.iloc[
    train_idx
].copy()

X_test = X.iloc[
    test_idx
].copy()

y_train = y.iloc[
    train_idx
].copy()

y_test = y.iloc[
    test_idx
].copy()

groups_train = groups.iloc[
    train_idx
]

groups_test = groups.iloc[
    test_idx
]

print(
    "Training rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

print(
    "\nTraining positive rate:",
    round(y_train.mean(), 4)
)

print(
    "Test positive rate:",
    round(y_test.mean(), 4)
)

Training rows: 136739
Test rows: 21051

Training positive rate: 0.0072
Test positive rate: 0.0041


In [17]:
# ============================================
# CELL 21 — Verify client separation
# ============================================

train_clients = set(
    groups_train
)

test_clients = set(
    groups_test
)

client_overlap = (
    train_clients
    .intersection(test_clients)
)

print(
    "Client overlap:",
    len(client_overlap)
)

assert len(client_overlap) == 0

print(
    "✓ Train/test client separation passed."
)

Client overlap: 0
✓ Train/test client separation passed.


In [18]:
# ============================================
# CELL 22 — Logistic Regression
# ============================================

logistic_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    )
])

logistic_model.fit(
    X_train,
    y_train
)

lr_scores = logistic_model.predict_proba(
    X_test
)[:, 1]

print("✓ Logistic Regression trained.")

✓ Logistic Regression trained.


## 6. Train Random Forest

Random Forest is used as the stronger nonlinear model.

It can capture interactions and nonlinear relationships between the decision-time features.

In [19]:
# ============================================
# CELL 23 — Random Forest
# ============================================

rf_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
])

rf_model.fit(
    X_train,
    y_train
)

rf_scores = rf_model.predict_proba(
    X_test
)[:, 1]

print("✓ Random Forest trained.")

✓ Random Forest trained.


## 7. Primary metric

The operational question is which observations should be reviewed first.

Therefore, Precision@K is used as the primary metric.

Precision@K answers:

> Among the top K observations prioritized by the method, what proportion are actually positive opportunities?

The ML-07 baseline and learned models are evaluated on the same held-out test observations.

In [20]:
# ============================================
# CELL 24 — Precision@K
# ============================================

def precision_at_k(y_true, scores, k):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    ranking = np.argsort(scores)[::-1]

    top_k = ranking[:k]

    return float(
        y_true[top_k].mean()
    )


K_VALUES = [10, 20, 50]

print(
    "Metrics:",
    [f"Precision@{k}" for k in K_VALUES]
)

Metrics: ['Precision@10', 'Precision@20', 'Precision@50']


## 8. Reproduce the ML-07 baseline

The ML-07 baseline uses the same rule that was defined in Week 4:

1. Impression volume must be at least the March median.
2. CTR must be below the median CTR for the page's average-position bucket.

The baseline is evaluated only on the ML-08 held-out test observations.

In [21]:
# ============================================
# CELL 25 — Reproduce ML-07 baseline
# ============================================

test_rows = march.iloc[test_idx].copy()

position_medians = (
    march
    .groupby(
        "position_bucket",
        observed=False
    )["ctr_pct"]
    .median()
)

test_rows["position_bucket_median_ctr"] = (
    test_rows["position_bucket"]
    .map(position_medians)
)

impression_threshold = (
    march["impressions"].median()
)

test_rows["baseline_prediction"] = (
    (
        test_rows["impressions"]
        >= impression_threshold
    )
    &
    (
        test_rows["ctr_pct"]
        < test_rows[
            "position_bucket_median_ctr"
        ]
    )
).astype(int)

baseline_scores = (
    test_rows["baseline_prediction"]
    .to_numpy()
)

print(
    "ML-07 impression threshold:",
    f"{impression_threshold:,.2f}"
)

print(
    "Baseline positive predictions:",
    baseline_scores.sum()
)

ML-07 impression threshold: 178.00
Baseline positive predictions: 215


In [22]:
# ============================================
# CELL 26 — Compare baseline and models
# ============================================

comparison_rows = []

methods = {
    "ML-07 Baseline": baseline_scores,
    "Logistic Regression": lr_scores,
    "Random Forest": rf_scores
}

for method, scores in methods.items():

    row = {
        "Model": method
    }

    for k in K_VALUES:

        row[f"Precision@{k}"] = precision_at_k(
            y_test,
            scores,
            k
        )

    comparison_rows.append(row)


comparison_df = pd.DataFrame(
    comparison_rows
)

display(comparison_df)

,Model,Precision@10,Precision@20,Precision@50
0,ML-07 Baseline,0.0,0.00,0.00
1,Logistic Regression,0.6,0.40,0.32
2,Random Forest,0.2,0.15,0.20


### Observed results

The comparison table above is generated directly from the notebook run.

The ML-07 baseline and learned models are evaluated on the same held-out test observations using the same Precision@K metrics.

No performance numbers are manually entered.

The best learned model is selected from the observed test results rather than assuming that the more complex model will perform better.

## 9. Errors and interpretation

I inspected false positives, false negatives, and model feature importance.

The error analysis is based on actual observations from the held-out test set rather than assumptions about why the model makes mistakes.

I also checked whether the strongest features are plausible and available at the decision moment.

A high aggregate score alone is not treated as proof that the model will improve CTR in production.

In [23]:
# ============================================
# CELL 29 — Select best learned model
# ============================================

PRIMARY_K = 50

PRIMARY_METRIC = (
    f"Precision@{PRIMARY_K}"
)

learned_results = comparison_df[
    comparison_df["Model"].isin([
        "Logistic Regression",
        "Random Forest"
    ])
].copy()

best_row = learned_results.loc[
    learned_results[
        PRIMARY_METRIC
    ].idxmax()
]

best_model_name = best_row["Model"]

baseline_score = comparison_df.loc[
    comparison_df["Model"]
    == "ML-07 Baseline",
    PRIMARY_METRIC
].iloc[0]

best_model_score = best_row[
    PRIMARY_METRIC
]

improvement = (
    best_model_score
    - baseline_score
)

print(
    "Best learned model:",
    best_model_name
)

print(
    "Baseline:",
    round(baseline_score, 4)
)

print(
    "Best model:",
    round(best_model_score, 4)
)

print(
    "Absolute improvement:",
    round(improvement, 4)
)

Best learned model: Logistic Regression
Baseline: 0.0
Best model: 0.32
Absolute improvement: 0.32


In [24]:
# ============================================
# CELL 30 — Error analysis
# ============================================

if best_model_name == "Random Forest":

    selected_scores = rf_scores
    selected_model = rf_model

else:

    selected_scores = lr_scores
    selected_model = logistic_model


selected_predictions = (
    selected_scores >= 0.5
).astype(int)


error_df = model_df.iloc[
    test_idx
].copy()

error_df["actual"] = (
    y_test.to_numpy()
)

error_df["prediction"] = (
    selected_predictions
)

error_df["model_score"] = (
    selected_scores
)


false_positives = error_df[
    (error_df["actual"] == 0)
    &
    (error_df["prediction"] == 1)
].sort_values(
    "model_score",
    ascending=False
)

false_negatives = error_df[
    (error_df["actual"] == 1)
    &
    (error_df["prediction"] == 0)
].sort_values(
    "model_score"
)


print("False positives:")
display(
    false_positives.head(3)
)

print("\nFalse negatives:")
display(
    false_negatives.head(3)
)

False positives:


,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,observed_days,first_report_date,last_report_date,position_bucket,...,clicks_april,ctr_april_pct,avg_position_april,observed_days_april,march_low_ctr,ctr_improved,is_opportunity,actual,prediction,model_score
42582,client_fef1a8f436438636,content_cdd93c30b9b06461,1347.0,0.0,0.0,1.168339,31,2026-03-01,2026-03-31,1-3,...,0.0,0.0,7.569904,26,True,False,0,0,1,0.998458
87951,client_fef1a8f436438636,content_8a29eeec466137f9,77.0,0.0,0.0,0.588790,9,2026-03-01,2026-03-09,1-3,...,0.0,0.0,6.366667,7,True,False,0,0,1,0.998456
123659,client_fef1a8f436438636,content_7aab39ab2c4b35e1,384.0,0.0,0.0,1.535554,9,2026-03-01,2026-03-15,1-3,...,0.0,0.0,6.768660,17,True,False,0,0,1,0.998270



False negatives:


,client_hash_id,content_hash_id,impressions,clicks,ctr_pct,avg_position,observed_days,first_report_date,last_report_date,position_bucket,...,clicks_april,ctr_april_pct,avg_position_april,observed_days_april,march_low_ctr,ctr_improved,is_opportunity,actual,prediction,model_score


In [25]:
# ============================================
# CELL 31 — Random Forest feature importance
# ============================================

rf_classifier = (
    rf_model
    .named_steps["model"]
)

feature_importance = pd.Series(
    rf_classifier.feature_importances_,
    index=FEATURE_COLUMNS
).sort_values(
    ascending=False
)

print("Top 10 Random Forest features:")

display(
    feature_importance
    .head(10)
    .to_frame("importance")
)

Top 10 Random Forest features:


,importance
position_inverse,0.422912
avg_position,0.393030
ctr_decimal,0.064925
ctr_pct,0.063174
impressions,0.018074
log_impressions,0.017539
observed_days,0.014086
clicks,0.003138
log_clicks,0.003123


## 10. Error interpretation

The strongest model features were inspected for plausibility and leakage.

The three strongest features were identified from the fitted model output rather than assumed in advance.

False-positive and false-negative examples were also inspected directly.

The final error patterns should be described from the observed examples. For example, if false positives cluster among pages with high impressions but CTR that is reasonable for their search position, this would indicate that the model may be over-prioritizing visibility.

If false negatives occur among pages with moderate impression volume but unusually strong CTR opportunity, this would indicate that the model may be under-prioritizing lower-volume pages.

These are examples of possible interpretations; the final submission should use only patterns actually observed in the test errors.

## Conclusion

The ML-08 experiment compares the ML-07 rule baseline with learned models using the same held-out test population and the same primary ranking metric.

The ML-07 baseline provides a simple decision-support queue based on high impression volume and low CTR relative to search position.

Logistic Regression provides an interpretable learned model, while Random Forest provides a nonlinear alternative.

The final model choice is based on the observed test results and error analysis. The result is treated as decision-support evidence rather than a guarantee of future CTR improvement.

The main limitation is that CTR is affected by factors that are not fully represented in the available data, including search intent, SERP features, brand effects, and other contextual factors.

In [26]:
# ============================================
# CELL 34 — Final self-check
# ============================================

print("========== ML-08 FINAL CHECK ==========")

print("Random seed:", RANDOM_STATE)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Test rows:",
    len(X_test)
)

print(
    "Training positive rate:",
    round(y_train.mean(), 4)
)

print(
    "Test positive rate:",
    round(y_test.mean(), 4)
)

print(
    "Client overlap:",
    len(
        set(groups_train)
        .intersection(
            set(groups_test)
        )
    )
)

print(
    "\nPrimary metric:",
    PRIMARY_METRIC
)

print(
    "Best learned model:",
    best_model_name
)

print("\nComparison table:")
display(comparison_df)

print("\nTop 3 Random Forest features:")
display(
    feature_importance
    .head(3)
)

assert len(
    set(groups_train)
    .intersection(
        set(groups_test)
    )
) == 0

print(
    "\n✓ Client leakage check passed."
)

print(
    "✓ ML-08 experiment completed."
)

========== ML-08 FINAL CHECK ==========
Random seed: 42
Training rows: 136739
Test rows: 21051
Training positive rate: 0.0072
Test positive rate: 0.0041
Client overlap: 0

Primary metric: Precision@50
Best learned model: Logistic Regression

Comparison table:


,Model,Precision@10,Precision@20,Precision@50
0,ML-07 Baseline,0.0,0.00,0.00
1,Logistic Regression,0.6,0.40,0.32
2,Random Forest,0.2,0.15,0.20



Top 3 Random Forest features:


,0
position_inverse,0.422912
avg_position,0.393030
ctr_decimal,0.064925



✓ Client leakage check passed.
✓ ML-08 experiment completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.